<a href="https://colab.research.google.com/github/shanmugt-hub/walsh/blob/main/notebooks/IT720_Assignment_4__ShanmuganathanT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### IT 720: NLP, Assignment 4 - A Tiny Transformer Language Model  175 Points

When you submit the assignment, make sure that you add your name to the file name.
When you think you are done, shut down the kernel one last time, restart it, and run your code from start to finish. Sometimes when you do this, you will discover that you still have errors. Leave the output in each cell to allow the grader to see it. If there is a bug that you cannot resolve, leave the error message in the output cell so the grader can see it.

#### Assignment Overview

For this assignment you will build your own language model using a simplified version of the original Transformer described in the Vaswani et. al. paper 'Attention is All You Need'. You do not have enough compute at home to build a "Large Language Model". However, even on a laptop you can build a model based on the alphabet rather than on words. This will reduce the computational load since there are far fewer alphabetic tokens than there are word tokens.

This model will probably take several hours to train. I suggest writing and debugging it using one epoch with a subset of the text for that epoch, and then, when your code works, training it for at least 2 epochs with the full text.

You will make many decisions for this model. Some of those decisions will be impacted by how much compute you have available. So when you first build your model, start small with only 1 transformer block with 2 attention heads, for example. Train it for one epoch to see how much time that takes, and make further decisions from there.

You can experiment with the number of transformer blocks, number of heads per block, embedding dimensions, and more. But be careful because a large model could easily require too much training time.  My own solution model has fewer than 500k trainable parameters and yet it still requires more than 1.5 hours per epoch.

After you train the model, you will also need to write an output function to generate text.  Detailed instructions for everything are given below.  

The purpose of this assignment is not to obtain impressive English output. Rather, the purpose is twofold:
1. Learn how to build a language model using the transformer architecture.
2. Gain some experience on the quality impact of various techniques for text generaton.


#### Rubric for the numbered sections below where you must write your own code, 175 points total

- 10 points: Task 1, Read and Preprocess the raw data
- 10 points: Task 2, Encode and Decode       
-  5 points: Task 3, set Hyperparameters
- 15 points: Task 4, Create training data
-  5 points: Task 5, causal mask function
- 35 points: Task 6, define TransformerBlock class  
- 35 points: Task 7, build Tiny Character Model    
- 15 points: Task 8, Compile and Summarize model     
- 10 points: Task 9, Train the model with at least 2 epochs
- 20 points: Task 10, Return character ID to generate  
- 15 points: Task 11, generate output text from seed

175 points total

In [1]:
# If you write your code in keras/tensorflow, then this cell has all of the imports you will need.
# You can write a solution using other packages, including PyTorch, if you prefer.
# However, many of my comments assume you are using Keras/TensorFlow, and it may be more
# work for you if you use PyTorch because some of those comments will be irrelevant.

import numpy as np
import os
import re
import tensorflow as tf
from   tensorflow.keras import layers, ops

print("TensorFlow version:", tf.__version__)
print("Keras version:",      tf.keras.__version__)

TensorFlow version: 2.19.0
Keras version: 3.13.2


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Task 1, Read and Preprocess: 10 points

# Read in a plain text novel (utf-8 format) /content/drive/MyDrive/AIML/IT720/Assignments/Datasets/Dracula.txt
# Bram Stoker's novel 'Dracula' (about 800k from the Gutenberg Project)
# has been given to you, but you can use any other novel if you prefer.
# You should preprocess the text by removing
# any non-printable characters that it may contain.
# Print about a thousand characters of the text both
# before the preprocessing, and after the preprocessing.
# You will probably see some words 'glued' together
# after the preprocessing, but this will not impact
# the model very much, though you are certainly free
# to do more sophisticated preprocessing if you wish.

###################### Insert Your Code Below ######################

file_path = '/content/drive/MyDrive/AIML/IT720/Assignments/Datasets/Dracula.txt'

# Read the raw text
with open(file_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()

print("--- Original Text (first 1000 characters) ---")
print(raw_text[:1000])
print("\n")

# Preprocess: remove non-printable characters
# Using a regex to keep only printable ASCII characters and common unicode characters (like smart quotes, em/en dashes, etc.)
# For simplicity, let's keep basic alphanumeric, punctuation, and whitespace.
# The problem description mentions removing 'non-printable characters'.
# A simple approach is to use string.printable or a regex that matches common printable characters.
# Let's use a regex to keep alphanumeric characters, common punctuation, and whitespace.

# Regex to match characters that are NOT alphanumeric, common punctuation, or whitespace
# This version keeps letters, numbers, basic punctuation, and whitespace.
# It also includes some common characters like smart quotes that might be in the text.
cleaned_text = re.sub(r'[^a-zA-Z0-9.,?!\-\"\':;()\s]+', '', raw_text)

# Replace multiple spaces with a single space to clean up any extra whitespace generated by character removal
cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

print("--- Preprocessed Text (first 1000 characters) ---")
print(cleaned_text[:1000])

# Store the cleaned text in a variable for later use
preprocessed_text = cleaned_text


--- Original Text (first 1000 characters) ---
How these papers have been placed in sequence will be made manifest in
the reading of them. All needless matters have been eliminated, so that
a history almost at variance with the possibilities of later-day belief
may stand forth as simple fact. There is throughout no statement of
past things wherein memory may err, for all the records chosen are
exactly contemporary, given from the standpoints and within the range
of knowledge of those who made them.




DRACULA




CHAPTER I

JONATHAN HARKER’S JOURNAL

(_Kept in shorthand._)


_3 May. Bistritz._--Left Munich at 8:35 P. M., on 1st May, arriving at
Vienna early next morning; should have arrived at 6:46, but train was an
hour late. Buda-Pesth seems a wonderful place, from the glimpse which I
got of it from the train and the little I could walk through the
streets. I feared to go very far from the station, as we had arrived
late and would start as near the correct time as possible. The
impre

In [4]:
# Task 2, Encode and Decode: 10 points

# Print the number of unique characters and those charactwers with an an appropriate identifying message.
# Create two python dictionaries, one that maps each unique character to an integer
# and the other that maps each integer back to its associated character

# Then write two functions:
# The 'encode' function takes a string of characters and returns an np.array of the integer encodings for the string.
# The 'decode' function does the inverse operation of returning the string that corresponds to an integer array input.
# Test your functions with the word 'vampire', encoding it to an array,
# then decoding it back to the original string.
# Print both the encoding and then the decoding to see if you get
# the same word from the decoder that you gave as input to the encoder.

###################### Insert Your Code Below ######################

# Get all unique characters from the preprocessed text
unique_chars = sorted(list(set(preprocessed_text)))
vocab_size = len(unique_chars)

print(f"Number of unique characters: {vocab_size}")
print(f"Unique characters: {''.join(unique_chars)}")

# Create character to integer mapping
char_to_int = {char: i for i, char in enumerate(unique_chars)}
# Create integer to character mapping
int_to_char = {i: char for i, char in enumerate(unique_chars)}

def encode(text_string):
    """Encodes a string into an array of integers."""
    return np.array([char_to_int[char] for char in text_string], dtype=np.int32)

def decode(encoded_array):
    """Decodes an array of integers back into a string."""
    return "".join([int_to_char[i] for i in encoded_array])

# Test the functions with the word 'vampire'
test_word = 'vampire'
encoded_word = encode(test_word)
decoded_word = decode(encoded_word)

print(f"\nOriginal word: '{test_word}'")
print(f"Encoded: {encoded_word}")
print(f"Decoded: '{decoded_word}'")


Number of unique characters: 72
Unique characters:  !(),-.0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz

Original word: 'vampire'
Encoded: [67 46 58 61 54 63 50]
Decoded: 'vampire'


Task 2, "Encode and Decode," focused on preparing the text for the model by converting characters into a numerical format and vice-versa. Here's what was done:

**Identify Unique Characters:**

All unique characters present in the preprocessed text were extracted, and the total number of unique characters (vocab_size) was determined.

**Create Mappings:**

Two dictionaries were created: char_to_int to map each unique character to a distinct integer ID, and int_to_char to map each integer ID back to its corresponding character.

**Define Encoding Function:**

An encode function was implemented to take a string and convert it into an np.array of these integer IDs.

**Define Decoding Function:**

A decode function was implemented to perform the inverse operation, converting an np.array of integer IDs back into a string.

**Test Functions:**

Both encode and decode functions were tested with the word 'vampire' to verify that they correctly encoded the word into a sequence of numbers and then decoded it back to the original word, ensuring proper functionality.

#### Visualization of the first few encoded sequences from Task 4

In [5]:
# Task 3, set Hyperparameters: 5 points

# Set all these hyperparameter variables as described
# You may change the variable names if you wish
# These will have a major impact on the quality of your model
# Be very careful, because some of these can result in very long
# training times if you choose a number that is large
# On the other hand, if you set them too small, the results can be quite bad!
# These are in alphabetical order

batchSize  = 64 # integer, minibatch size
dimModel   = 128 # integer, model dimensionality (char embeddings and intermediate vectors)
epochs     = 2 # integer
ffDim      = 512 # integer, neurons of feedforward net within transformer
maxLength  = 129 # integer, the number of rows for the position embeddings, but maxLength
               # needs to be at least 1 larger than windowSize to avoid error when
               # making a prediction
numHeads   = 2 # integer, number of attention heads per transformer block
numLayers  = 1 # integer, defines number of transformer blocks to be built by the language model
vocabSize  = 72 # integer, the number of unique characters that are in the dictionaries of the previous cell
windowSize = 128 # integer, largest context length for the character model,
               # i.e. at most, model will look at previous windowSize characters

# Note: the windowSize (context) will have a large impact on how
# much time your model will take to train. But the benefit of longer
# window size is better output!


In [6]:
# Task 4, Create training data: 15 points

# For each encoded character, the target character for the model to predict
# will be the very next character of the text.
# So first, use your encoder to encode the entire preprocessed text.
# Then you will need to create two np.array variables, one for the model
# inputs and the other for the model targets (outputs).

# The model will look at sequences of encoded characters of windowSize
# that you just set in the previous cell. Then it will try to predict the
# next character from the encoded text.
# Thus, each row of inputs and targets should have windowSize columns.
# However, because you are always predicting the next character,
# the targets will be offset from the inputs by exactly one character,
# i.e. if windowSize is 5, the a row of the inputs might be:
# [29 63 71  0 68] and the corresponding row of the targets would look like:
# [63 71  0 68 56], assuming that 56 is the very next encoding after 68 in the text

# After creating these two np.arrays, call three print statements:
# The first will print the shapes of the two arrays, which should have the same shapes.
# The second will be the first row of characters from the inputs, and
# the third print will be the corresponding row from from the target (outputs),
# and you should visually verify that the list of targets is offset by 1 from the inputs.

###################### Insert Your Code Below ######################

# Encode the entire preprocessed text
encoded_text = encode(preprocessed_text)

# Calculate the number of sequences that can be created
# Each sequence will be `windowSize` long, and targets are shifted by 1
# So, if `encoded_text` length is L, and `windowSize` is W:
# Inputs will go from index 0 to L - W - 1
# Targets will go from index 1 to L - W
num_sequences = len(encoded_text) - windowSize

# Initialize input and target arrays
inputs = np.zeros((num_sequences, windowSize), dtype=np.int32)
targets = np.zeros((num_sequences, windowSize), dtype=np.int32)

# Populate input and target arrays
for i in range(num_sequences):
    inputs[i] = encoded_text[i : i + windowSize]
    targets[i] = encoded_text[i + 1 : i + windowSize + 1]

# Print shapes of the arrays
print(f"Shape of inputs: {inputs.shape}")
print(f"Shape of targets: {targets.shape}")

# Print the first row of inputs and targets for verification
print(f"\nFirst row of inputs (encoded): {inputs[0]}")
print(f"First row of targets (encoded): {targets[0]}")

# Decode and print the first row for visual verification (optional but helpful)
print(f"First row of inputs (decoded): '{decode(inputs[0])}'")
print(f"First row of targets (decoded): '{decode(targets[0])}'")


Shape of inputs: (828710, 128)
Shape of targets: (828710, 128)

First row of inputs (encoded): [27 60 68  0 65 53 50 64 50  0 61 46 61 50 63 64  0 53 46 67 50  0 47 50
 50 59  0 61 57 46 48 50 49  0 54 59  0 64 50 62 66 50 59 48 50  0 68 54
 57 57  0 47 50  0 58 46 49 50  0 58 46 59 54 51 50 64 65  0 54 59  0 65
 53 50  0 63 50 46 49 54 59 52  0 60 51  0 65 53 50 58  6  0 20 57 57  0
 59 50 50 49 57 50 64 64  0 58 46 65 65 50 63 64  0 53 46 67 50  0 47 50
 50 59  0 50 57 54 58 54]
First row of targets (encoded): [60 68  0 65 53 50 64 50  0 61 46 61 50 63 64  0 53 46 67 50  0 47 50 50
 59  0 61 57 46 48 50 49  0 54 59  0 64 50 62 66 50 59 48 50  0 68 54 57
 57  0 47 50  0 58 46 49 50  0 58 46 59 54 51 50 64 65  0 54 59  0 65 53
 50  0 63 50 46 49 54 59 52  0 60 51  0 65 53 50 58  6  0 20 57 57  0 59
 50 50 49 57 50 64 64  0 58 46 65 65 50 63 64  0 53 46 67 50  0 47 50 50
 59  0 50 57 54 58 54 59]
First row of inputs (decoded): 'How these papers have been placed in sequence will be made 

In [7]:
# Display the first 5 encoded input sequences
print("First 5 Encoded Input Sequences:")
for i in range(5):
    print(f"Sequence {i+1}: {inputs[i]}")
print("\n")

# Display the first 5 decoded input sequences
print("First 5 Decoded Input Sequences:")
for i in range(5):
    print(f"Sequence {i+1}: '{decode(inputs[i])}'")
print("\n")

# Display the first 5 encoded target sequences
print("First 5 Encoded Target Sequences:")
for i in range(5):
    print(f"Sequence {i+1}: {targets[i]}")
print("\n")

# Display the first 5 decoded target sequences
print("First 5 Decoded Target Sequences:")
for i in range(5):
    print(f"Sequence {i+1}: '{decode(targets[i])}'")


First 5 Encoded Input Sequences:
Sequence 1: [27 60 68  0 65 53 50 64 50  0 61 46 61 50 63 64  0 53 46 67 50  0 47 50
 50 59  0 61 57 46 48 50 49  0 54 59  0 64 50 62 66 50 59 48 50  0 68 54
 57 57  0 47 50  0 58 46 49 50  0 58 46 59 54 51 50 64 65  0 54 59  0 65
 53 50  0 63 50 46 49 54 59 52  0 60 51  0 65 53 50 58  6  0 20 57 57  0
 59 50 50 49 57 50 64 64  0 58 46 65 65 50 63 64  0 53 46 67 50  0 47 50
 50 59  0 50 57 54 58 54]
Sequence 2: [60 68  0 65 53 50 64 50  0 61 46 61 50 63 64  0 53 46 67 50  0 47 50 50
 59  0 61 57 46 48 50 49  0 54 59  0 64 50 62 66 50 59 48 50  0 68 54 57
 57  0 47 50  0 58 46 49 50  0 58 46 59 54 51 50 64 65  0 54 59  0 65 53
 50  0 63 50 46 49 54 59 52  0 60 51  0 65 53 50 58  6  0 20 57 57  0 59
 50 50 49 57 50 64 64  0 58 46 65 65 50 63 64  0 53 46 67 50  0 47 50 50
 59  0 50 57 54 58 54 59]
Sequence 3: [68  0 65 53 50 64 50  0 61 46 61 50 63 64  0 53 46 67 50  0 47 50 50 59
  0 61 57 46 48 50 49  0 54 59  0 64 50 62 66 50 59 48 50  0 68 54 57 57
  0

In [8]:
# Task 5, causal mask function: 5 points

# Define a function called causal_attention_mask
# Its output should be a boolean array to prevent your model
# from looking into the future, as we discuss in Week 8.
# An example of this function can be found in the
# keras documentation at this URL:
# https://keras.io/examples/generative/text_generation_with_miniature_gpt/

# However, depending on which version of Keras you are using,
# AND whether or not you are accessing Keras from within TensorFlow,
# you may not need to define a custom masking function because
# some versions of MultiHeadAttention have a boolean call argument
# that will automatically do the masking for you.
# To be protected against these versioning differences,
# you can just use the custom causal attention mask at the previous URL.
# The following URL shows the masking argument used in Keras 3:
# https://keras.io/api/layers/attention_layers/multi_head_attention/
# See also Task 6 below where the causal masking is used.

###################### Insert Your Code Below ######################

@tf.function
def causal_attention_mask(batch_size, n_dest, n_src, dtype):
    """
    Masks future tokens for multi-head attention.
    Taken from Keras example: https://keras.io/examples/generative/text_generation_with_miniature_gpt/
    """
    i = ops.arange(n_dest)[:, None]
    j = ops.arange(n_src)
    m = i >= j
    mask = ops.cast(m, dtype)
    mask = ops.reshape(mask, [1, n_dest, n_src])
    mult = ops.concatenate(
        [ops.expand_dims(batch_size, -1), ops.array([1, 1], dtype=tf.int32)],
        axis=0,
    )
    return tf.tile(mask, mult) # Changed ops.tile to tf.tile


This code defines the causal_attention_mask function, which is crucial for building Transformer models, especially for text generation. Let me break it down for you:

@tf.function: This decorator is from TensorFlow and it's used to compile the Python function into a highly performant, callable TensorFlow graph. This helps speed up the execution, especially during training.

causal_attention_mask(batch_size, n_dest, n_src, dtype): This function takes four arguments:

batch_size: The number of sequences being processed in parallel.
n_dest: The length of the target sequence (e.g., the query sequence).
n_src: The length of the source sequence (e.g., the key/value sequence).
dtype: The data type for the mask, typically tf.bool.
Creating Index Tensors: i = ops.arange(n_dest)[:, None] creates a column vector of indices from 0 to n_dest-1. Similarly, j = ops.arange(n_src) creates a row vector of indices from 0 to n_src-1.

Generating the Causal Logic (m = i >= j): This is the core of the causal mask. It compares each element of the i tensor with each element of the j tensor. The result m is a 2D boolean matrix where m[row, col] is True if row >= col, and False otherwise. This effectively creates a lower triangular matrix, ensuring that a token at position row can only attend to tokens at positions col that are less than or equal to row (i.e., current and past tokens), preventing it from 'seeing' future tokens.

Casting and Reshaping: The boolean mask m is converted to the specified dtype and then reshaped to [1, n_dest, n_src]. The initial 1 in the shape is a placeholder for the batch dimension, allowing it to be broadcast across multiple batches.

Tiling for Batch Dimension: mult = ops.concatenate(...) creates a tensor mult that specifies how many times to repeat the mask along each dimension. It essentially says to repeat the mask batch_size times along the first (batch) dimension, and once along the n_dest and n_src dimensions.

Returning the Mask (tf.tile(mask, mult)): Finally, tf.tile replicates the single mask batch_size times, resulting in a mask of shape [batch_size, n_dest, n_src]. Each sequence in the batch will receive the same causal mask, effectively blocking attention to future tokens.

In simple terms, this function generates a mask that tells the attention mechanism which parts of the input sequence it's allowed to look at. For generative models, it's crucial that a token can only look at itself and previous tokens, never future ones, to ensure it's truly predicting the next token based on past context. The use of tf.tile instead of ops.tile was a fix to resolve a specific TensorFlow internal error related to symbolic tensor iteration.

The causal_attention_mask function is central to how the TransformerBlock ensures the model adheres to a causal structure, which is vital for generative tasks like text prediction. Here's how it's used:

Calling the Mask Function: Inside the call method of the TransformerBlock class, the causal_attention_mask function is invoked.

Determining Dimensions: The batch_size and seq_len (sequence length) are extracted from the input tensor inputs that the TransformerBlock receives. These dimensions are then passed to the causal_attention_mask function.

Generating the Mask: The causal_attention_mask function, decorated with @tf.function, generates a boolean tensor (or a tensor of the specified dtype) of shape [batch_size, seq_len, seq_len]. This mask has False (or a very large negative number, depending on implementation details in MultiHeadAttention) entries for positions where a token should not attend to future tokens, effectively creating a lower-triangular-like structure.

Applying to MultiHeadAttention: This generated causal_mask is then passed directly as the attention_mask argument to the self.att (the MultiHeadAttention layer) within the TransformerBlock. The MultiHeadAttention layer uses this mask during its calculation to zero out (or set to negative infinity before softmax) the attention weights corresponding to future tokens.

In essence: For each position in the sequence that the TransformerBlock is processing, the causal_attention_mask ensures that the attention mechanism only considers information from the current position and all preceding positions, preventing the model from 'cheating' by looking at future characters it's trying to predict.

In [9]:
# Task 6, define TransformerBlock class: 35 points

# Define a TransformerBlock class subclassing it from the Keras Layer class
# You will find a very useful example of this in the Keras documentation example
# at this URL that you could modify as needed for this assignment:
# https://keras.io/examples/generative/text_generation_with_miniature_gpt/

# Make sure your TransformerBlock includes the following:
# 1. a self reference to the Keras MultiHeadAttention class
#    passing it the number of heads and the key dimensionality (model dimensionality)
# 2. a self reference to the feed forward sequential network that is inside
#    the transformer blocks as described in the Vaswani paper and that
#    we discuss in Week 8. Keep this FFNN small, with only two
#    dense layers, one with the number of feed forward neurons defined above
#    in Task 3 for Hyperparameters, and also using an activation function of your choice,
#    and the other layer using the number of neurons as defined by the model dimensionality
#    also defined above in Task 3.
# 3. Two normalization layers (Keras LayerNormalization layer)
# 4. Optional use of two Dropout layers (Keras Dropout layer)
# 5. Appropriate call arguments and associated code to help construct the model architecture.
#    Again, refer to this URL regarding how the TransformerBlock contributes to the architecture:
#    https://keras.io/examples/generative/text_generation_with_miniature_gpt/

#    Also, if you use the custom causal_attention_mask above,
#    you will need to use it in the call method to set a mask
#    which you will pass to MultiHeadAttention with its call argument
#    called 'attention_mask' which is necessary if you are using a custom mask.

#    Finally, if you are using dropout, be sure to look at the documentation
#    for the 'training' argument of MultiHeadAttention:
#    https://keras.io/api/layers/attention_layers/multi_head_attention/
#    Using dropout, you'll need to use: training=False in the call method
#    to pass it to MultiHeadAttention for generation.
#    If you choose NOT to use dropout, you don't need to worry about
#    training=False at all. Even if you do use dropout, you don't
#    need to worry about setting training=True because that will happen
#    automatically when the model.fit method is invoked.

###################### Insert Your Code Below ######################

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential(
            [
                layers.Dense(ff_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training):
        # Multi-Head Attention
        input_shape = ops.shape(inputs)
        batch_size = input_shape[0]
        seq_len = input_shape[1]
        causal_mask = causal_attention_mask(
            batch_size, seq_len, seq_len, tf.bool
        )

        attn_output = self.att(inputs, inputs, attention_mask=causal_mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        # Feed Forward Network
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)


In [10]:
# Task 7, build tiny character model: 35 points

# Define a tiny transformer character model
# You can find a variety of ways to do this on the web,for example:
# a.  Your model can be subclassed from the Keras Model class.
#     An example of this that also includes a Python function
#     to finalize the model can be found here:
#     https://keras.io/examples/nlp/masked_language_modeling/
#     This is the preferred way because of the complex architecture
#     of transformer models.
# b.  Another approach is possible that uses the functional API.
#     You can see the functional API being used in the URL below
#     buried inside of a standard Python function definition.
#     https://keras.io/examples/generative/text_generation_with_miniature_gpt/
#     But again, I recommend using the subclassing approach.

# Whichever way you decide to build your model, make sure it includes:
# 1. An embedding layer (Keras Embedding class) for the character-based vocabulary.
#    The number of rows would be the number of characters
#    (vocabulary size in a word-based language model),
#    which you should have already determined in Task 2 above.
#    The number of columns would be the model dimensionality from Task 3.
# 2. Another embedding layer to learn the character token positions
#    The number of rows would be the maxLength from Task 3 and
#    the columns should, of course, again be the model dimensionality.
# 3. 1 or 2 transformer blocks (more if you have a lot compute and time)
# 4. A layer normalization layer followed by the output layer, which
#    should return the output logits

# IMPORTANT NOTE: In most classifier models you would have an output layer
# that uses a softmax activation function to produce a probability distribution
# over the categories. DO NOT DO THAT HERE. The softmax needs to appear in your
# text generation function that you will define in Task 8 below.
# See additional comments for that Task below.

###################### Insert Your Code Below ######################

class TinyCharacterModel(tf.keras.Model):
    def __init__(self, vocab_size, max_length, embed_dim, num_heads, ff_dim, num_layers):
        super().__init__()
        self.token_emb = layers.Embedding(vocab_size, embed_dim) # Character embedding
        self.pos_emb = layers.Embedding(max_length, embed_dim) # Positional embedding
        self.transformer_blocks = [
            TransformerBlock(embed_dim, num_heads, ff_dim) for _ in range(num_layers)
        ]
        self.layernorm = layers.LayerNormalization(epsilon=1e-6)
        self.output_layer = layers.Dense(vocab_size) # Output logits

    def call(self, inputs, training):
        seq_len = ops.shape(inputs)[-1]
        positions = ops.arange(0, seq_len)

        # Get embeddings for tokens and positions
        token_embeddings = self.token_emb(inputs)
        position_embeddings = self.pos_emb(positions)

        # Combine embeddings
        x = token_embeddings + position_embeddings

        # Pass through transformer blocks
        for transformer_block in self.transformer_blocks:
            x = transformer_block(x, training=training) # Fix: Pass 'training' as a keyword argument

        # Final layer normalization and output layer
        x = self.layernorm(x)
        return self.output_layer(x)

# Instantiate the model
tinyModel = TinyCharacterModel(vocabSize, maxLength, dimModel, numHeads, ffDim, numLayers)


In [11]:
# Task 8, Compile and Summarize model: 15 points

# You should use the SparseCategoricalCrossentropy loss function

# IMPORTANT NOTE 1: The Keras documentation for SparseCategoricalCrossentropy is here.
#    https://keras.io/api/losses/probabilistic_losses/#sparsecategoricalcrossentropy-class
#    If you look at its 'Arguments' section you will see the following:
#      "from_logits: Whether y_pred is expected to be a logits tensor.
#       By default, we assume that y_pred encodes a probability distribution."
#    In the preceding cell I told you not to use a softmax in your output layer.
#    Because of that, you need to change this default argument value to:
#    from_logits=True when you reference SparseCategoricalCrossentropy
# I will give you additional information about this below when you define
# the text generation function.

# IMPORTANT NOTE 2: Depending on how you defined the model above,
# you may need to uncomment the 'BUILD' code that you see in this
# cell. Run this cell first with the BUILD code commented out.
# If the model summary shows that there are 0 trainable parameters,
# then uncomment the code and re-run. Again, depending on how you
# define the model, it may not be able to know in advance the shapes
# at different layers, preventing it from building the model.
# The BUILD code forces it to build the model, and then you
# will see the number of your trainable parameters.
# In my model with the BUILD code, it shows: Trainable params: 486,236

# You can use any optimizer that you wish

###################### Insert Your Code Below ######################

# Compile the model
tinyModel.compile(
    optimizer="adam", # Using Adam optimizer
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

# Explicitly build the model with a known input shape (removing this line as it's not fully effective)
# tinyModel.build(input_shape=(None, windowSize))

# BUILD the model by calling it once (if needed, depends on how you built the model)
# Uncomment this code only if you see the model summary below, but it shows
# that there are 0 trainable parameters.

buildVar = tf.zeros((1, windowSize), dtype=tf.int32)
dummyVar = tinyModel(buildVar, training=False)

tinyModel.summary()


Model: "tiny_character_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (1, 128, 128)          │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (128, 128)             │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ ?                      │       264,192 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_2           │ (1, 128, 128)          │           256 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (1, 128, 72)           │         9,288 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 299,464 (1.14 MB)

 Trainable params: 299,464 (1.14 MB)

 Non-trainable params: 0 (0.00 B)

ask 8, handles the compilation and summarization of your TinyCharacterModel. Let's break it down:

Model Compilation: The tinyModel.compile() method configures the model for training.

optimizer="adam": This sets the optimizer to 'Adam', a popular and effective optimization algorithm for neural networks.
loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True): This is crucial. SparseCategoricalCrossentropy is used because your target labels are integers (character IDs) and your model's output layer returns raw logits (unnormalized scores), not probabilities. Setting from_logits=True tells the loss function to apply a softmax activation internally to these logits before calculating the cross-entropy loss.
metrics=["accuracy"]: This specifies that during training and evaluation, the model should report its accuracy.
Explicit Model Building: The lines involving buildVar and dummyVar (buildVar = tf.zeros((1, windowSize), dtype=tf.int32) and dummyVar = tinyModel(buildVar, training=False)) are there to explicitly build the model.

When you define a Keras model by subclassing tf.keras.Model (like TinyCharacterModel), the layers are often not instantiated or assigned their weights until the model sees its first input. Calling the model with a dummy input (buildVar) forces all layers to be built and their parameters created, even before model.fit is called. This helps prevent errors related to uninitialized layers.
training=False is passed to the dummy call because this is just for building, not for actual training.
Model Summary: tinyModel.summary() prints a concise summary of the model's architecture, including:

The name and type of each layer.
The output shape of each layer.
The number of parameters (weights and biases) in each layer.
A total count of trainable and non-trainable parameters in the entire model.
This step ensures that your model is properly configured and that all its components are correctly initialized before you proceed with training.

In [12]:
# Task 9, Train the model with at least 2 epochs: 10 points

###################### Insert Your Code Below ######################

print("Starting model training...")

# Create a tf.data.Dataset from inputs and targets
dataset = tf.data.Dataset.from_tensor_slices((inputs, targets))
dataset = dataset.batch(batchSize)

history = tinyModel.fit(
    dataset,
    epochs=epochs
)

print("Model training finished.")


Starting model training...
Epoch 1/2
12949/12949 ━━━━━━━━━━━━━━━━━━━━ 114s 6ms/step - accuracy: 0.4167 - loss: 1.9616
Epoch 2/2
12949/12949 ━━━━━━━━━━━━━━━━━━━━ 52s 4ms/step - accuracy: 0.5056 - loss: 1.6351
Model training finished.


In [13]:
# Task 10, Return character ID to generate: 20 points

# Define one function that returns the numeric ID
# of the next character to generate.

# Refer to the following sections in Jurafsky and Martin (August 2025)
# Softmax of the logits: discussed in J&M section 7.4.3
# Temperature Sampling:  described in J&M section 7.4.3
# Top-k Sampling:        described in J&M section 8.6.1
# Top-p Sampling:        described in J&M section 8.6.2

# You can find examples of both top-k and top-p code on the web,
# but they will likely require modification.

# IMPORTANT NOTE: You must use temperature sampling and softmax,
# and you can choose to implement EITHER Top-k OR Top-p,
# but not both because they don't work well with each other.

# You could write your own softmax in numpy or you could use
# TensorFlow's tf.nn.softmax or scipy's version:
# https://docs.scipy.org/doc//scipy-1.16.1/reference/generated/scipy.special.softmax.html

###################### Insert Your Code Below ######################

def sample_next_char_id(logits, temperature=1.0, top_k=0, top_p=0.0):
    """
    Samples the next character ID from logits using temperature, Top-k, or Top-p sampling.

    Args:
        logits (tf.Tensor): The unnormalized log probabilities (logits) for the next character.
                            Expected shape (vocab_size,).
        temperature (float): Controls the randomness of predictions. Higher values make output more random.
        top_k (int): If > 0, only consider the top_k most likely tokens.
        top_p (float): If between 0 and 1, consider the smallest set of tokens whose cumulative
                       probability exceeds top_p.

    Returns:
        int: The sampled character ID.
    """
    # Apply temperature
    logits = logits / temperature

    if top_k > 0:
        # Keep only top_k logits, set others to a very small negative number
        values, _ = tf.nn.top_k(logits, k=top_k)
        min_value = tf.reduce_min(values)
        logits = tf.where(logits < min_value, tf.ones_like(logits) * -1e10, logits)
    elif 0 < top_p < 1.0:
        # Sort logits and compute probabilities
        sorted_logits = tf.sort(logits, direction='DESCENDING')
        sorted_indices = tf.argsort(logits, direction='DESCENDING')

        probabilities = tf.nn.softmax(sorted_logits)
        cumulative_probs = tf.cumsum(probabilities)

        # Find the number of tokens to keep (index of the first token that pushes cumulative_probs over top_p)
        cutoff_index = tf.where(cumulative_probs >= top_p)[0][0]
        num_tokens_to_keep = cutoff_index + 1

        # Create a mask for the original logits to keep only the top_p ones
        indices_to_keep = sorted_indices[:num_tokens_to_keep]

        # Create an empty tensor for filtered logits
        filtered_logits = tf.ones_like(logits) * -1e10
        # Update only the logits corresponding to the indices_to_keep
        logits = tf.tensor_scatter_nd_update(
            filtered_logits,
            tf.expand_dims(indices_to_keep, axis=-1),
            tf.gather(logits, indices_to_keep)
        )

    # Sample next character ID from the (potentially filtered) logits
    # tf.random.categorical expects logits for sampling
    sampled_id = tf.random.categorical(tf.expand_dims(logits, 0), num_samples=1)[0, 0].numpy()

    return sampled_id


This code defines the sample_next_char_id function, which is a crucial part of generating text from your language model. Its main purpose is to select the next character ID based on the raw predictions (logits) from the model, incorporating various sampling strategies to control the randomness and quality of the generated text.

Let's break down how it works:

Function Signature: sample_next_char_id(logits, temperature=1.0, top_k=0, top_p=0.0)

logits (tf.Tensor): These are the unnormalized raw scores from your model for each possible character in the vocabulary. The higher the logit for a character, the more likely the model thinks it is to be the next character.
temperature (float): Controls the 'creativity' or randomness. A temperature of 1.0 means no change. Higher values (e.g., 1.5) make the distribution flatter, increasing the probability of less likely characters, leading to more diverse but potentially less coherent text. Lower values (e.g., 0.5) make the distribution sharper, favoring more likely characters and producing more predictable text.
top_k (int): If greater than 0, only the top_k characters with the highest logits are considered for sampling. All other characters are assigned a very low probability.
top_p (float): If between 0 and 1, this method (also known as nucleus sampling) considers the smallest set of characters whose cumulative probability mass exceeds top_p. This dynamically adjusts the number of characters considered, which can be more flexible than top_k.
Apply Temperature: The first step logits = logits / temperature scales the logits. If temperature is high, logits become smaller, making the probability distribution flatter after softmax. If temperature is low, logits become larger (in magnitude), making the distribution sharper.

Top-k Sampling (Conditional):

If top_k > 0, it uses tf.nn.top_k to find the top_k largest logits and their corresponding indices.
It then creates a new logits tensor where only these top_k logits are kept, and all other logits are set to a very small negative number (-1e10), effectively giving them a near-zero probability after softmax.
Top-p Sampling (Conditional):

If 0 < top_p < 1.0, it first sorts the logits in descending order and calculates their softmax probabilities.
It then computes the cumulative sum of these sorted probabilities.
It finds the smallest number of characters whose cumulative probability sum is just above top_p.
Similar to Top-k, it creates a new logits tensor where only the characters within this 'nucleus' are kept, and others are suppressed with a very small negative number.
Important Note: The problem statement indicates you should implement either Top-k or Top-p, not both, as they don't generally combine well.
Final Sampling: Regardless of which filtering method (Top-k or Top-p) was applied, the function then uses tf.random.categorical to sample one character ID from the (potentially modified) logits. This function takes logits and performs a weighted random choice based on the implicit probabilities.

Return Value: The function returns the integer ID of the sampled next character.

In essence, this function provides a flexible way to introduce controlled randomness into the text generation process, allowing you to balance between generating highly probable and more diverse sequences.

In [14]:
# Task 11, generate output text from seed: 15 points

# Write a generate function that takes your trained model,
# and a context seed - a short piece of text that sets up a context
# for the model to create a continuation - using the
# sampling function that you defined in Task 10.
# You will also need to pass it any parameters needed by your
# sampling function defined in the previous cell because the
# text generator will call that function.

# Then call the generator function to generate text.
# The first thing you see in the output should be your 'seed'
# text followed by text generated by the model.

###################### Insert Your Code Below ######################

def generate_text(model, seed_text, num_generate=500, temperature=1.0, top_k=0, top_p=0.0):
    """
    Generates text using the trained model, a seed text, and sampling parameters.

    Args:
        model (tf.keras.Model): The trained TinyCharacterModel.
        seed_text (str): The starting text for generation.
        num_generate (int): The number of characters to generate.
        temperature (float): Controls randomness of predictions. Higher values make output more random.
        top_k (int): If > 0, only consider the top_k most likely tokens.
        top_p (float): If between 0 and 1, consider the smallest set of tokens whose cumulative
                       probability exceeds top_p.

    Returns:
        str: The generated text.
    """
    # Encode the seed text
    input_eval = encode(seed_text)
    # Expand dimensions to match model input shape (batch_size, sequence_length)
    input_eval = tf.expand_dims(input_eval, 0)

    # List to store generated characters
    text_generated = []

    # Print the seed text
    print(f"Seed: '{seed_text}'")

    # Here, `temperature` is used directly in `sample_next_char_id`.
    # The loop iterates `num_generate` times to generate `num_generate` characters.
    for i in range(num_generate):
        # Model expects input of shape (batch_size, windowSize)
        # We need to ensure input_eval always has windowSize characters.
        # If input_eval is longer than windowSize, take the last windowSize characters.
        if input_eval.shape[1] > windowSize:
            input_to_model = input_eval[:, -windowSize:]
        else:
            input_to_model = input_eval

        # Get predictions from the model (logits for the next character)
        # The model outputs logits for each character in the input_to_model sequence.
        # We are interested in the prediction for the *last* character in the sequence.
        predictions = model(input_to_model, training=False)[:, -1, :]

        # Sample the next character ID using the provided sampling function
        predicted_id = sample_next_char_id(predictions[0], temperature=temperature, top_k=top_k, top_p=top_p)

        # Convert the predicted ID to a character and append to generated text
        text_generated.append(int_to_char[predicted_id])

        # Update the input_eval for the next prediction
        input_eval = tf.concat([input_eval, tf.expand_dims([predicted_id], 0)], axis=-1)

    return seed_text + "".join(text_generated)

# --- Example Usage ---
# Generate text with some seed and parameters
seed = "The old castle" # Make sure the seed text uses characters present in unique_chars
# If the seed_text length is less than windowSize, the model will pad it internally with its positional embeddings.
# For better results, especially initially, a seed longer than windowSize might not be ideal because of how input is handled.
# It's better to provide a seed that is not longer than windowSize.
# Let's adjust the seed length if it's too long for the windowSize.

if len(seed) > windowSize:
    seed = seed[-windowSize:] # Take the last part of the seed

generated_novel_text = generate_text(
    tinyModel,
    seed_text=seed,
    num_generate=500, # Generate 500 characters
    temperature=0.8,  # A bit less random than 1.0
    top_k=50          # Consider top 50 likely characters
)

print("\nGenerated Text:")
print(generated_novel_text)


Seed: 'The old castle'



Generated Text:
The old castle oir rest; outh at our so out our me been had him. No! I fol took of lashed this ligh and of mist me of they was fallined living of round as nothes matche. The is astowo to me that time was be is and his to have be of eyes litting of their gried some of lame far been ass the was that that the had that He snime, but the were in was I wash of know it wolves eye. For it a ever bath, so the same in incrief word; but is the be and which was the wom in the heave the reser of a black purgess far which 


This code defines the generate_text function, which is the final step in this assignment: generating output text from your trained language model. Let's break down its components:

Function Definition: generate_text(model, seed_text, num_generate=500, temperature=1.0, top_k=0, top_p=0.0)

model: The tf.keras.Model (your TinyCharacterModel) that has been trained.
seed_text: A string that provides the initial context for the model to start generating text from.
num_generate: The total number of characters the function should generate after the seed_text.
temperature, top_k, top_p: These parameters are passed directly to the sample_next_char_id function (defined in Task 10) to control the randomness and diversity of the generated characters.
Initialization:

input_eval = encode(seed_text): The seed_text is first converted into its numerical (encoded) representation using the encode function from Task 2.
input_eval = tf.expand_dims(input_eval, 0): The encoded seed is then expanded to have a batch dimension, making its shape (1, sequence_length) to match the model's expected input format.
text_generated = []: An empty list is initialized to store the IDs of the characters generated by the model.
Generation Loop: The code then enters a loop that runs num_generate times, generating one character at each step:

Context Window Management: if input_eval.shape[1] > windowSize: input_to_model = input_eval[:, -windowSize:] else: input_to_model = input_eval. This is crucial. The model was trained to predict the next character based on a context of windowSize characters. To ensure consistency, this part of the code always feeds the last windowSize characters (or fewer, if input_eval is shorter than windowSize) from input_eval to the model. This means the model always operates on a fixed-size context window.
Prediction: predictions = model(input_to_model, training=False)[:, -1, :]. The input_to_model (the current context window) is passed to the tinyModel. training=False ensures that dropout layers are disabled during generation. The model outputs logits for each character in the input_to_model sequence. We are only interested in the logits for the last character of this sequence, as that's what the model is predicting from.
Sampling: predicted_id = sample_next_char_id(predictions[0], temperature=temperature, top_k=top_k, top_p=top_p). The logits for the next character are then passed to the sample_next_char_id function (from Task 10) along with the chosen temperature, top_k, or top_p parameters. This function selects the numerical ID of the next character.
Appending Generated Character: text_generated.append(int_to_char[predicted_id]). The predicted_id is converted back to its character form using int_to_char (from Task 2) and added to the text_generated list.
Updating Input Context: input_eval = tf.concat([input_eval, tf.expand_dims([predicted_id], 0)], axis=-1). The newly predicted character's ID is appended to the input_eval tensor. This updated input_eval serves as the context for predicting the next character in the subsequent loop iteration.
Return Value: After the loop completes, the function concatenates the original seed_text with all the newly text_generated characters and returns the complete generated string.

Example Usage: The final part of the cell demonstrates how to use this generate_text function with a sample seed text ("The old castle") and specific sampling parameters (temperature=0.8, top_k=50). It also includes a check to truncate the seed if it's longer than windowSize, which is good practice for the model's expected input.

Analyzing the quality of the generated text, starting with the seed: "The old castle".

Generated Text: "The old castlent as will the came we can of our I kneed. When was desary more expossed been and abright ask feel time! as the Bist yof to the end fash, and enter, look over. His look and of Madam Mr. More gan more safe had be battle lay away nothind from the engle of the our hand; and to snow, or a do reyes suffice: Franginst of dark In the sring returnneed wise; and expain on Look a sen a stall atther day earies, sursed; and our own it was placeed all on bothout me and of the track. His look the to mall est"

Quality Analysis:

The text quality is quite low, which is expected given the small model size (numLayers=1, numHeads=2, dimModel=128) and limited training (only 2 epochs). Here's a breakdown:

Coherence: The text is largely incoherent. While it starts with the seed, the generated sentences quickly devolve into a jumble of words with little to no semantic meaning or logical flow. It doesn't tell a story or form continuous ideas.
Grammar and Syntax: Grammatical structure is almost non-existent beyond very short phrases. Sentence boundaries are often unclear, and many phrases are nonsensical.
Spelling: There are numerous spelling errors (e.g., "castlent", "desary", "expossed", "nothind", "engle", "sring returnneed", "expain", "atther day earies", "sursed", "placeed", "bothout"). This indicates the model hasn't fully learned character-level word forms, or the limited vocabulary and training struggle to capture them.
Relevance to Seed: It begins directly with the seed, but quickly loses any connection. While some words might be tangentially related to the 'Dracula' novel (e.g., "Madam Mr.", "dark"), they don't form a coherent narrative related to the seed.
Punctuation: Punctuation is haphazardly placed and often incorrect, contributing to the overall lack of readability.
Conclusion:

This model, with its current configuration and minimal training, is primarily generating plausible sequences of characters and word fragments, but it has not yet learned to produce syntactically correct or semantically coherent English text. To improve the quality, you would typically need:

More Training Epochs: This is the most crucial factor. Training for more epochs would allow the model to learn more complex patterns and improve both grammatical structure and vocabulary.
Larger Model: Increasing numLayers, numHeads, dimModel, and ffDim could enhance the model's capacity to learn intricate language relationships, though this requires significantly more computational resources and time.
Adjusted Sampling Parameters: While temperature=0.8 and top_k=50 introduce randomness, with a poorly trained model, they can sometimes lead to more erratic output. Experimenting with different temperature values (e.g., lower to make it more deterministic) or top_k/top_p values might yield slightly different results, but fundamental improvements require more training.

To significantly improve your model's text generation quality, especially given that it's a character-level model with limited computational resources, here are the primary strategies:

Increase Training Epochs: This is usually the most impactful change you can make without altering the model architecture. Training for more epochs allows the model to see the entire dataset multiple times, leading to better learning of patterns, grammar, and character sequences. The current model was trained for only 2 epochs, which is very minimal for a language model.

Increase Model Capacity (Larger Model): While being mindful of compute limits, increasing the model's capacity allows it to learn more complex relationships.

numLayers: Add more TransformerBlocks (e.g., 2, 3, or even more if your hardware allows). Each additional layer deepens the network, enabling it to capture hierarchical features.
numHeads: Increase the number of attention heads (e.g., from 2 to 4 or 8). More attention heads allow the model to jointly attend to different parts of the sequence, focusing on different aspects of the input simultaneously.
dimModel (Embedding Dimension): Increase the dimModel (e.g., from 128 to 256 or 512). A larger embedding dimension provides a richer representation for each character and position, and for the intermediate vectors within the Transformer blocks.
ffDim (Feed-Forward Dimension): Increase the size of the feed-forward network's hidden layer (e.g., from 512 to 1024). This provides more computational power within each Transformer block.
Caution: Be very careful with increasing these. Each increase drastically multiplies the number of trainable parameters and, consequently, training time and memory requirements. Start small and incrementally increase while monitoring resource usage.
Optimize Hyperparameters: Fine-tuning the hyperparameters can yield better results.

Learning Rate: Experiment with different learning rates for your Adam optimizer. A learning rate schedule (e.g., decreasing it over time) can also be beneficial.
batchSize: While 64 is a reasonable starting point, experimenting with slightly larger or smaller batch sizes might affect convergence speed and generalization.
Refine Sampling Parameters: After a better-trained model, you can fine-tune your sampling parameters (temperature, top_k, top_p) for desired output characteristics.

Lower Temperature: For more coherent and predictable text, try a lower temperature (e.g., 0.5-0.7). This makes the model 'less adventurous'.
Adjust top_k / top_p: If using top_k, try values like 20, 30, or 40. For top_p, values like 0.85 or 0.9. These values truncate less probable options, focusing the generation on more plausible continuations.
Given that this is an assignment focused on building the architecture, increasing the epochs and incrementally testing slightly larger model parameters (numLayers, dimModel, ffDim, numHeads) would be the most direct ways to see tangible improvements within the scope of your work.